# Fine-tune ModernBERT — Hallucination Detection in Tool Calling

Token classifier on top of either:
- `KRLabsOrg/lettucedect-large-modernbert-en-v1` — already fine-tuned on RAGTruth (recommended; transfer learning).
- `answerdotai/ModernBERT-large` — base model from scratch.

Input: `[CLS] context [SEP] question [SEP] answer [SEP]`. Per-token binary labels (1 = hallucinated). Context/question tokens are masked out of the loss (label=-100).

Upload `combined_train.jsonl`, `combined_val.jsonl`, `combined_test.jsonl` (drop in `/content/` or run upload cell). **Requires GPU**.

In [ ]:
import subprocess, sys, os, importlib.util

NEEDED = ("transformers", "accelerate", "torch")
missing = [p for p in NEEDED if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers>=4.48", "accelerate", "datasets"])
    if "google.colab" in sys.modules:
        print("Restarting kernel…")
        os.kill(os.getpid(), 9)
else:
    print("Already installed.")


## Configuration

`BASE_MODEL` choice:
- `KRLabsOrg/lettucedect-large-modernbert-en-v1` — transfer from RAGTruth (faster convergence).
- `answerdotai/ModernBERT-large` — train from scratch (slower, more flexible).

In [ ]:
from pathlib import Path

CANDIDATE_DIRS = [
    Path("/content"),
    Path("/kaggle/input/halluc-toolace"),
    Path("data"),
    Path("."),
]
REQUIRED = ["combined_train.jsonl", "combined_val.jsonl", "combined_test.jsonl"]
DATA_DIR = next((d for d in CANDIDATE_DIRS if all((d / f).exists() for f in REQUIRED)), None)

if DATA_DIR is None:
    print("Files not found in standard locations. Upload via the cell below.")
else:
    print(f"Using DATA_DIR = {DATA_DIR}")

BASE_MODEL = "KRLabsOrg/lettucedect-large-modernbert-en-v1"
OUTPUT_DIR = "/content/checkpoints"
# T4 (16 GB) fits: ModernBERT-large + max_len=2048 + bs=2 + gradient_checkpointing.
# Effective batch = BATCH_SIZE * GRAD_ACCUM_STEPS.
MAX_LENGTH = 2048
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 2
GRADIENT_CHECKPOINTING = True
LR = 1e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3
WARMUP_RATIO = 0.05
EVAL_EVERY_STEPS = 200
SEED = 42
CLEAN_OVERSAMPLE = 4   # replicate clean samples K times to reach ~25% clean in train

import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}, mem {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
if DATA_DIR is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_DIR = Path("/content")
    except ImportError:
        raise RuntimeError("Set DATA_DIR manually if not in Colab.")
for f in REQUIRED:
    print(f"  {f}: {(DATA_DIR / f).exists()}")


## Helpers: data encoding + metrics

In [ ]:
import json
from dataclasses import dataclass

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def char_spans_to_token_labels(answer_offsets, spans):
    labels = [0] * len(answer_offsets)
    for span in spans:
        s, e = int(span["start"]), int(span["end"])
        for i, (a, b) in enumerate(answer_offsets):
            if a == b: continue
            if a < e and b > s:
                labels[i] = 1
    return labels

@dataclass
class Encoded:
    input_ids: list
    attention_mask: list
    labels: list
    answer_token_offsets: list
    answer_token_indices: list

def encode_sample(sample, tokenizer, max_length=4096):
    enc_ctx = tokenizer(sample["context"],  add_special_tokens=False, truncation=False)
    enc_q   = tokenizer(sample["query"],    add_special_tokens=False, truncation=False)
    enc_a   = tokenizer(sample["output"],   add_special_tokens=False, return_offsets_mapping=True, truncation=False)
    cls_id, sep_id = tokenizer.cls_token_id, tokenizer.sep_token_id
    ids = [cls_id] + enc_ctx["input_ids"] + [sep_id] + enc_q["input_ids"] + [sep_id]
    answer_start = len(ids)
    ids += enc_a["input_ids"]
    answer_end = len(ids)
    ids += [sep_id]
    if len(ids) > max_length:
        excess = len(ids) - max_length
        ctx_len = len(enc_ctx["input_ids"])
        keep = max(0, ctx_len - excess)
        new_ctx = enc_ctx["input_ids"][:keep]
        ids = [cls_id] + new_ctx + [sep_id] + enc_q["input_ids"] + [sep_id] + enc_a["input_ids"] + [sep_id]
        answer_start = 1 + len(new_ctx) + 1 + len(enc_q["input_ids"]) + 1
        answer_end = answer_start + len(enc_a["input_ids"])
    attn = [1] * len(ids)
    labels = [-100] * len(ids)
    a_labels = char_spans_to_token_labels(enc_a["offset_mapping"], sample["hallucination_labels"])
    for i, lab in enumerate(a_labels):
        labels[answer_start + i] = lab
    return Encoded(ids, attn, labels, enc_a["offset_mapping"],
                   list(range(answer_start, answer_end)))

# Metrics
def _char_set(spans):
    out = set()
    for s in spans:
        out.update(range(int(s["start"]), int(s["end"])))
    return out

def span_micro_f1(samples, pred_spans):
    o = p = g = 0
    for s, ps in zip(samples, pred_spans):
        gs = _char_set(s["hallucination_labels"]); pset = _char_set(ps)
        o += len(gs & pset); p += len(pset); g += len(gs)
    pr = o/p if p else 0.0; re = o/g if g else 0.0
    f1 = 2*pr*re/(pr+re) if (pr+re) else 0.0
    return pr, re, f1

def example_f1(samples, pred_spans):
    tp = fp = fn = 0
    for s, ps in zip(samples, pred_spans):
        gold = bool(s["hallucination_labels"]); pred = bool(ps)
        if gold and pred: tp += 1
        elif gold: fn += 1
        elif pred: fp += 1
    pr = tp/(tp+fp) if (tp+fp) else 0.0
    re = tp/(tp+fn) if (tp+fn) else 0.0
    f1 = 2*pr*re/(pr+re) if (pr+re) else 0.0
    return pr, re, f1

train_samples = read_jsonl(DATA_DIR / "combined_train.jsonl")
val_samples   = read_jsonl(DATA_DIR / "combined_val.jsonl")
test_samples  = read_jsonl(DATA_DIR / "combined_test.jsonl")

# Oversample clean samples in train (clean = empty hallucination_labels).
if CLEAN_OVERSAMPLE > 1:
    clean = [s for s in train_samples if not s["hallucination_labels"]]
    hallu = [s for s in train_samples if s["hallucination_labels"]]
    train_samples = hallu + clean * CLEAN_OVERSAMPLE
    pct = 100 * len(clean) * CLEAN_OVERSAMPLE / len(train_samples)
    print(f"Oversampled clean × {CLEAN_OVERSAMPLE}: train = {len(hallu)} hallu + {len(clean)*CLEAN_OVERSAMPLE} clean ({pct:.0f}% clean)")
print(f"train={len(train_samples)}, val={len(val_samples)}, test={len(test_samples)}")


## Load model + tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, num_labels=2,
                                                        ignore_mismatched_sizes=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Loaded {BASE_MODEL} on {device}, params={sum(p.numel() for p in model.parameters())/1e6:.0f}M")


## Train

Uses HuggingFace `Trainer`. ~30-60 min on T4 for 3 epochs on ~10k samples.

In [ ]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

class HallucinationDS(Dataset):
    def __init__(self, samples, tokenizer, max_length):
        self.samples = samples; self.tok = tokenizer; self.max_length = max_length
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        e = encode_sample(self.samples[idx], self.tok, self.max_length)
        return {"input_ids": e.input_ids, "attention_mask": e.attention_mask, "labels": e.labels}

train_ds = HallucinationDS(train_samples, tokenizer, MAX_LENGTH)
val_ds   = HallucinationDS(val_samples,   tokenizer, MAX_LENGTH)

def collate(batch):
    ml = max(len(b["input_ids"]) for b in batch)
    pad_id = tokenizer.pad_token_id or 0
    def pad(seq, val): return seq + [val] * (ml - len(seq))
    return {
        "input_ids":      torch.tensor([pad(b["input_ids"], pad_id) for b in batch], dtype=torch.long),
        "attention_mask": torch.tensor([pad(b["attention_mask"], 0)  for b in batch], dtype=torch.long),
        "labels":         torch.tensor([pad(b["labels"], -100)        for b in batch], dtype=torch.long),
    }

import inspect
ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    try:
        model.config.use_cache = False
    except Exception:
        pass

args_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    eval_steps=EVAL_EVERY_STEPS,
    save_steps=EVAL_EVERY_STEPS,
    save_total_limit=2,
    logging_steps=50,
    bf16=torch.cuda.is_available(),
    seed=SEED,
    report_to=[],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)
# Pick whichever name this transformers version recognizes for eval/save strategy.
if "eval_strategy" in ta_params:
    args_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in ta_params:
    args_kwargs["evaluation_strategy"] = "steps"
args_kwargs["save_strategy"] = "steps"

args = TrainingArguments(**args_kwargs)

trainer_kwargs = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=collate)
trainer_params = set(inspect.signature(Trainer.__init__).parameters)
# transformers ≥4.46 renamed `tokenizer` to `processing_class`.
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
trainer.train()


## Evaluate on combined_test (+ per-type subsets)

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

THRESHOLD = 0.5
model.eval()

def predict_spans(sample):
    e = encode_sample(sample, tokenizer, MAX_LENGTH)
    inputs = {
        "input_ids":      torch.tensor([e.input_ids],      dtype=torch.long, device=device),
        "attention_mask": torch.tensor([e.attention_mask], dtype=torch.long, device=device),
    }
    with torch.no_grad():
        out = model(**inputs)
    probs = torch.softmax(out.logits[0], dim=-1)[:, 1]
    a_probs = probs[e.answer_token_indices].tolist()
    text = sample["output"]
    spans = []; cs = ce = None; cmax = 0.0
    for (a, b), p in zip(e.answer_token_offsets, a_probs):
        if a == b: continue
        if p > THRESHOLD:
            if cs is None: cs = a
            ce = b; cmax = max(cmax, p)
        else:
            if cs is not None:
                spans.append({"start": cs, "end": ce, "text": text[cs:ce], "confidence": cmax})
                cs = ce = None; cmax = 0.0
    if cs is not None:
        spans.append({"start": cs, "end": ce, "text": text[cs:ce], "confidence": cmax})
    return spans

test_preds = [predict_spans(s) for s in tqdm(test_samples, desc="Inference (combined_test)")]

def filter_subset(label):
    s, p = [], []
    for x, y in zip(test_samples, test_preds):
        if not x["hallucination_labels"]:
            s.append(x); p.append(y); continue
        if x["hallucination_labels"][0].get("type") == label:
            s.append(x); p.append(y)
    return s, p

subsets = {
    "Combined (all + clean)":      (test_samples, test_preds),
    "Type 1 + clean":              filter_subset("Type1_Contradiction"),
    "Type 2 + clean":              filter_subset("Type2_Overgeneration"),
    "Type 3 + clean":              filter_subset("Type3_MissingTool"),
}
rows = []
for name, (s, p) in subsets.items():
    sp, sr, sf = span_micro_f1(s, p)
    ep, er, ef = example_f1(s, p)
    rows.append({"Split": name, "N": len(s),
                 "Span P": round(sp, 3), "Span R": round(sr, 3), "Span F1": round(sf, 3),
                 "Ex P": round(ep, 3), "Ex R": round(er, 3), "Ex F1": round(ef, 3)})
    print(f"{name} (N={len(s)}): span P/R/F1 = {sp:.3f}/{sr:.3f}/{sf:.3f} | ex P/R/F1 = {ep:.3f}/{er:.3f}/{ef:.3f}")
pd.DataFrame(rows)


## Save model

In [ ]:
FINAL_DIR = "/content/checkpoints/best"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Saved to {FINAL_DIR}")

# Optional: zip for download
import shutil
shutil.make_archive("/content/best_model", "zip", FINAL_DIR)
print("Zipped to /content/best_model.zip — download via the file panel.")
